# Set the Working Directory 
Always set the current working directory to the root of the repo so you can import the helper functions 

In [2]:
import os
# Change to the directory where the script is located to be able to import local modules
os.chdir("..")

In [3]:
from pathlib import Path
cwd = Path.cwd()
print(cwd)

h:\My Drive\Courses\MasterThesis\Code\NonLinear_FEMSM


# Libraries 

In [5]:
from utils.checkpoints import load_checkpoint
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

gamma_value  = gamma_a

base = "_".join(format_param_key_val(k, v) for k, v in exps[gamma_model_dict[gamma_value]][1].items() if k not in ["alpha","omega"])

ckpt_dir = checkpoint_dir/f"{base}"

def iteration_from_path(path):
    # matches the digits before “.pt” after the last underscore
    m = re.search(r"_(\d+)\.pt$", path)
    return int(m.group(1)) if m else -1

# grab *all* .pt files in this folder
pattern = str(Path(ckpt_dir) / "*.pt")
ckpts = sorted(glob.glob(pattern), key=iteration_from_path)

latest = ckpts[-1]
# load the trained the model

model, optimizer, history, start_epoch = load_checkpoint(
        checkpoint_path = latest,
        model_class = SeqGPLVM,
        optimizer_class = torch.optim.Adam,
        device=device
    )
param_hist = history['param_hist']
actual_params = history['actual_params']
loss_list = history['loss_list']

df = exps[gamma_model_dict[gamma_value]][0]
metadata_2 = exps[gamma_model_dict[gamma_value]][1]

# Loss 

In [ ]:
import matplotlib.pyplot as plt
last_n_step = 14000
fig,ax = plt.subplots(1,2,figsize = (10,3))
ax[0].plot(loss_list)
ax[1].plot(range(len(loss_list)- last_n_step,len(loss_list)) ,loss_list[-last_n_step:])
plt.show()

# Params

In [ ]:
params = set([item.split(".")[-1] for item in param_hist.keys()])
params

In [ ]:
from utils.inspectors import plot_param_history
#param_hist_copy = param_hist.copy()
#param_hist_copy["Z.q_mu"] = param_hist["Z.q_mu"][-100:]
# param_hist = param_hist for raw values and param_hist = actual_params for actual values after transformation.
# ls_num only matters for plotting the length_scale (s)
key = "raw_lengthscale"
fig = plot_param_history(param_hist = actual_params, key= key, ls_num=2)
fig